# Blinded pharmacogenomics study - 110 cases, 21 genes

Genotype -> diplotype -> CPIC phenotype -> CPIC Level A recommendation.

Diplotypes are **called** from the CPIC allele-definition tables (`pgx_pipeline.call_diplotypes`),
phenotypes come from the CPIC `diplotype` table and recommendations from the CPIC `recommendation`
table, with the supplied guideline corpus used where CPIC publishes no machine-readable rows
(warfarin, ondansetron, peginterferon alfa).

Run order: `fetch_cpic_api.py` (API snapshot) -> this notebook / `pgx_pipeline.py` -> `make_methods_note.py`.

In [1]:
import json, os, collections, importlib.util
import pandas as pd

HERE = os.path.abspath(".")
cases = json.load(open(os.path.join(HERE, "cases_blinded.json")))
corpus = json.load(open(os.path.join(HERE, "cpic_guideline_corpus.json")))
meta = json.load(open(os.path.join(HERE, "cpic_api_snapshot", "_fetch_meta.json")))

print(f"{len(cases)} cases | {len(corpus['genes'])} genes in corpus (schema {corpus['schema_version']}, "
      f"generated {corpus['generated_at_utc']})")
print(f"CPIC API snapshot fetched {meta['fetched_at_utc']}, latest change_log entry {meta['change_log_latest_date']}")
pd.DataFrame(collections.Counter((c['gene'], c['drug']) for c in cases).items(),
             columns=['gene_drug', 'n_cases']).sort_values('n_cases', ascending=False).head(10)

110 cases | 21 genes in corpus (schema v3.1, generated 2026-05-10T04:40:15Z)
CPIC API snapshot fetched 2026-09-16T14:27:50.593227+00:00, latest change_log entry 2026-08-03


,gene_drug,n_cases
0,"(CYP2D6, codeine)",5
5,"(CYP2C19, clopidogrel)",5
7,"(CYP2C19, citalopram)",5
6,"(CYP2C19, voriconazole)",5
8,"(CYP2C19, omeprazole)",5
4,"(CYP2D6, amitriptyline)",4
3,"(CYP2D6, paroxetine)",4
1,"(CYP2D6, tamoxifen)",4
13,"(CYP2B6, efavirenz)",4
14,"(DPYD, fluorouracil)",4


## 1. Load the pipeline and inspect the variant -> allele definitions it will use

In [2]:
spec = importlib.util.spec_from_file_location("pgx", os.path.join(HERE, "pgx_pipeline.py"))
pgx = importlib.util.module_from_spec(spec); spec.loader.exec_module(pgx)

# CPIC plus-strand base for each case rsID, per allele (the basis of every star-allele call)
rows = []
for gene, rsid in [("CYP2D6", "rs3892097"), ("CYP2C19", "rs4244285"), ("CYP2C19", "rs12248560"),
                   ("CYP2C9", "rs1057910"), ("CYP3A5", "rs776746"), ("TPMT", "rs1800460"),
                   ("NUDT15", "rs116855232"), ("SLCO1B1", "rs4149056"), ("DPYD", "rs3918290")]:
    loc = pgx.RSID2LOC[gene][rsid]
    for name, dd in pgx.ALLELE_DEF[gene].items():
        if loc in dd and name in ("*1", "*2", "*3", "*3B", "*4", "*5", "*17", "Reference",
                                  "c.1905+1G>A (*2A)"):
            rows.append([gene, rsid, pgx.SL[loc]['name'], name, "".join(sorted(dd[loc])),
                         "reference" if dd[loc] == pgx.ref_base(gene, loc) else "variant"])
pd.DataFrame(rows, columns=["gene", "rsid", "cpic_variant", "allele", "plus_strand_base", "vs_reference"])

,gene,rsid,cpic_variant,allele,plus_strand_base,vs_reference
0,CYP2D6,rs3892097,1847G>A,*1,C,reference
1,CYP2D6,rs3892097,1847G>A,*4,T,variant
2,CYP2C19,rs4244285,19154G>A,*2,A,variant
3,CYP2C19,rs12248560,-806C>T,*4,CT,variant
4,CYP2C19,rs12248560,-806C>T,*17,T,variant
5,CYP2C9,rs1057910,42614A>C,*1,A,reference
6,CYP2C9,rs1057910,42614A>C,*3,C,variant
7,CYP3A5,rs776746,6981A>G,*1,T,reference
8,CYP3A5,rs776746,6981A>G,*3,C,variant
9,TPMT,rs1800460,460G>A,*1,C,reference


## 2. Run the pipeline (writes the answers file and the lookup tables)

In [3]:
results = pgx.main()
answers = pd.read_csv(os.path.join(HERE, "claude_science_answers.csv"), dtype=str)
audit = pd.read_csv(os.path.join(HERE, "case_audit.csv"), dtype=str)
assert list(answers.case_id) == [c['id'] for c in cases]
answers.head(12)

cases=110 abstained=5 lethal_class=29
abstentions:
  4f19dd3f5283 UGT1A1       irinotecan       no CPIC Level A recommendation text exists for UGT1A1-irinotecan; phenotype is reported but the recommendation
  912145392abc UGT1A1       irinotecan       no CPIC Level A recommendation text exists for UGT1A1-irinotecan; phenotype is reported but the recommendation
  6eeeaffed889 UGT1A1       irinotecan       no CPIC Level A recommendation text exists for UGT1A1-irinotecan; phenotype is reported but the recommendation
  e55c1b0dd1a0 CYP4F2       warfarin         CPIC warfarin dosing is algorithm-based and indexed on VKORC1 and CYP2C9; CYP4F2 genotype alone does not yield
  807ece818520 CYP4F2       warfarin         CPIC warfarin dosing is algorithm-based and indexed on VKORC1 and CYP2C9; CYP4F2 genotype alone does not yield


,case_id,diplotype,phenotype,recommendation,abstained
0,70dc9c9110f3,*4/*4,Poor Metabolizer,"codeine: Avoid codeine use because of possibility of diminished analgesia. If opioid use is warranted, consider a non-tramadol opioid.",false
1,b07cff8c99c0,*1/*4,Intermediate Metabolizer,"codeine: Use codeine label recommended age- or weight-specific dosing. If no response and opioid use is warranted, consider a non-tramadol opioid.",false
2,ce34b1a18a3d,*1/*1,Normal Metabolizer,codeine: Use codeine label recommended age- or weight-specific dosing.,false
3,74597c5503ec,*1/*1xN (N>=2),Ultrarapid Metabolizer,"codeine: Avoid codeine use because of potential for serious toxicity. If opioid use is warranted, consider a non-tramadol opioid.",false
4,3d0e97b86ad0,*1/*4,Intermediate Metabolizer,"codeine: Use codeine label recommended age- or weight-specific dosing. If no response and opioid use is warranted, consider a non-tramadol opioid.",false
5,f9e4a13551fd,*4/*4,Poor Metabolizer,tamoxifen: Recommend alternative hormonal therapy such as an aromatase inhibitor for postmenopausal women or aromatase inhibitor along with ovarian function suppression in premenopausal women given that these approaches are superior to tamoxifen regardless of CYP2D6 genotype (PMID 26211827) and based on knowledge that CYP2D6 poor metabolizers switched from tamoxifen to anastrozole do not have an increased risk of recurrence (PMID 23213055).,false
6,9d75134030af,*1/*4,Intermediate Metabolizer,"tamoxifen: Consider hormonal therapy such as an aromatase inhibitor for postmenopausal women or aromatase inhibitor along with ovarian function suppression in premenopausal women, given that these approaches are superior to tamoxifen regardless of CYP2D6 genotype (PMID 26211827). If aromatase inhibitor use is contraindicated, consideration should be given to use a higher but FDA approved tamoxifen dose (40 mg/day)(PMID 27226358). Avoid CYP2D6 strong to weak inhibitors.",false
7,987d7bd14933,*1/*1,Normal Metabolizer,tamoxifen: Avoid moderate and strong CYP2D6 inhibitors. Initiate therapy with recommended standard of care dosing (tamoxifen 20 mg/day).,false
8,61741d10f238,*1/*1xN (N>=2),Ultrarapid Metabolizer,tamoxifen: Avoid moderate and strong CYP2D6 inhibitors. Initiate therapy with recommended standard of care dosing (tamoxifen 20 mg/day).,false
9,c0cb8d7ad685,*4/*4,Poor Metabolizer,ondansetron: Insufficient evidence demonstrating clinical impact based on CYP2D6 genotype. Initiate therapy with recommended starting dose.,false


## 3. Abstentions

In [4]:
audit.loc[audit.abstained == 'true', ['case_id', 'gene', 'drug', 'phenotype', 'abstain_reason']]

,case_id,gene,drug,phenotype,abstain_reason
76,4f19dd3f5283,UGT1A1,irinotecan,Poor Metabolizer,no CPIC Level A recommendation text exists for UGT1A1-irinotecan; phenotype is reported but the recommendation is abstained
77,912145392abc,UGT1A1,irinotecan,Intermediate Metabolizer,no CPIC Level A recommendation text exists for UGT1A1-irinotecan; phenotype is reported but the recommendation is abstained
78,6eeeaffed889,UGT1A1,irinotecan,Normal Metabolizer,no CPIC Level A recommendation text exists for UGT1A1-irinotecan; phenotype is reported but the recommendation is abstained
102,e55c1b0dd1a0,CYP4F2,warfarin,n/a - CPIC assigns no phenotype term to CYP4F2; rs2108622 T/T (*3/*3),CPIC warfarin dosing is algorithm-based and indexed on VKORC1 and CYP2C9; CYP4F2 genotype alone does not yield a dose recommendation (no CYP4F2 phenotype table and no CYP4F2 recommendation rows in the CPIC API; the supplied corpus excerpt contains no CYP4F2 dosing statement)
103,807ece818520,CYP4F2,warfarin,n/a - CPIC assigns no phenotype term to CYP4F2; rs2108622 C/C (*1/*1),CPIC warfarin dosing is algorithm-based and indexed on VKORC1 and CYP2C9; CYP4F2 genotype alone does not yield a dose recommendation (no CYP4F2 phenotype table and no CYP4F2 recommendation rows in the CPIC API; the supplied corpus excerpt contains no CYP4F2 dosing statement)


## 4. Lethal-class cases

Standard dosing expected to cause severe, irreversible or life-threatening harm, or a life-threatening therapeutic failure.

In [5]:
leth = audit[audit.lethal_class == 'true']
print(f"{len(leth)} of {len(audit)} cases")
leth[['case_id', 'gene', 'drug', 'diplotype', 'phenotype', 'harm_mechanism']]

29 of 110 cases


,case_id,gene,drug,diplotype,phenotype,harm_mechanism
3,74597c5503ec,CYP2D6,codeine,*1/*1xN (N>=2),Ultrarapid Metabolizer,morphine overdose: fatal respiratory depression
16,d456dcf2b837,CYP2D6,amitriptyline,*4/*4,Poor Metabolizer,"tricyclic accumulation: seizures, cardiac arrhythmia"
20,ca9e14feca53,CYP2C19,clopidogrel,*2/*2,Poor Metabolizer,"loss of antiplatelet effect: stent thrombosis, MI, death (ACS/PCI)"
21,97d4e4559a13,CYP2C19,clopidogrel,*1/*2,Intermediate Metabolizer,"loss of antiplatelet effect: stent thrombosis, MI, death (ACS/PCI)"
25,00b16db7406d,CYP2C19,voriconazole,*2/*2,Poor Metabolizer,"voriconazole accumulation: hepatotoxicity, neurotoxicity"
28,1fb04aa9e16c,CYP2C19,voriconazole,*1/*17,Rapid Metabolizer,subtherapeutic exposure: failure of therapy for invasive fungal infection
29,0c21c969a5c4,CYP2C19,voriconazole,*17/*17,Ultrarapid Metabolizer,subtherapeutic exposure: failure of therapy for invasive fungal infection
40,97c3baebed4e,CYP2C9,warfarin,*3/*3,Poor Metabolizer,over-anticoagulation: major/fatal haemorrhage
43,e85ecfa887cb,CYP2C9,phenytoin,*3/*3,Poor Metabolizer,phenytoin accumulation: severe neurotoxicity
56,b94a43e574db,DPYD,fluorouracil,c.1905+1G>A (*2A)/c.1905+1G>A (*2A),Poor Metabolizer,"fluoropyrimidine overdose: fatal myelosuppression, mucositis, neurotoxicity"


## 5. Sensitivity of the calls to the panel-completeness assumption

Every SNV case is re-called with untyped loci treated as *unknown* instead of reference. Cases
whose compatible-phenotype set grows are the ones where the answer leans on the panel convention.

In [6]:
sens = audit[audit.phenotype_if_untyped_loci_unknown.notna() &
             (audit.phenotype_if_untyped_loci_unknown.fillna('') != '')].copy()
sens['stable'] = sens.apply(
    lambda x: set(x['phenotype_if_untyped_loci_unknown'].split('; ')) == {x['phenotype']}, axis=1)
print(f"{(~sens.stable).sum()} of {len(sens)} SNV-based calls are assumption-dependent")
sens.loc[~sens.stable, ['case_id', 'gene', 'drug', 'diplotype', 'phenotype',
                        'phenotype_if_untyped_loci_unknown']]

15 of 77 SNV-based calls are assumption-dependent


,case_id,gene,drug,diplotype,phenotype,phenotype_if_untyped_loci_unknown
4,3d0e97b86ad0,CYP2D6,codeine,*1/*4,Intermediate Metabolizer,Indeterminate; Intermediate Metabolizer; Poor Metabolizer
40,97c3baebed4e,CYP2C9,warfarin,*3/*3,Poor Metabolizer,Indeterminate; Poor Metabolizer
41,9b5e3093ab14,CYP2C9,warfarin,*1/*3,Intermediate Metabolizer,Indeterminate; Intermediate Metabolizer
43,e85ecfa887cb,CYP2C9,phenytoin,*3/*3,Poor Metabolizer,Indeterminate; Poor Metabolizer
44,c9726ef5c899,CYP2C9,phenytoin,*1/*3,Intermediate Metabolizer,Indeterminate; Intermediate Metabolizer
46,b0db8db6df72,CYP2C9,celecoxib,*3/*3,Poor Metabolizer,Indeterminate; Poor Metabolizer
47,6b95f7ee2785,CYP2C9,celecoxib,*1/*3,Intermediate Metabolizer,Indeterminate; Intermediate Metabolizer
55,240c62812cbc,CYP2B6,efavirenz,*1/*4,Rapid Metabolizer,Intermediate Metabolizer; Rapid Metabolizer
70,d1a12c4a16c4,SLCO1B1,simvastatin,*5/*5,Poor Function,Indeterminate; Poor Function; Possible Decreased Function
71,51a382a7ca5c,SLCO1B1,simvastatin,*1/*5,Decreased Function,Decreased Function; Indeterminate


## 6. Summary by gene and by recommendation source

In [7]:
g = audit.groupby('gene').agg(cases=('case_id', 'size'),
                             abstained=('abstained', lambda s: (s == 'true').sum()),
                             lethal_class=('lethal_class', lambda s: (s == 'true').sum()))
print(audit.recommendation_source.str.split(r' \(').str[0].value_counts().to_string())
print()
print(audit.cpic_pair_level.value_counts().to_string())
g

recommendation_source
CPIC data API recommendation table                          95
CPIC warfarin guideline                                      7
CPIC ondansetron/CYP2D6 guideline table, supplied corpus     3
CPIC pair table + supplied corpus                            3
CPIC IFNL3/peginterferon guideline, supplied corpus          2

cpic_pair_level
A          108
Retired      2


,cases,abstained,lethal_class
gene,,,
CFTR,2,0,0
CYP2B6,4,0,0
CYP2C19,20,0,5
CYP2C9,9,0,2
CYP2D6,20,0,2
CYP3A5,3,0,0
CYP4F2,2,2,0
DPYD,7,0,5
G6PD,6,0,4
